# Create PGW WRF intermediate files

In [1]:
from pathlib import Path

import numpy as np
import pywinter.winter as pyw
import xarray as xr
import xesmf as xe

In [2]:
src_dir = Path("/data7/khanhdn/wrf/WPS/runs/era5_test/")
dst_dir = "/data7/khanhdn/wrf/WPS/runs/era5_test_pgw/"
wrf_inter_prefix = "ERA5"

In [3]:
wps_inter_filepaths = list(src_dir.glob(f"{wrf_inter_prefix}:*"))

In [4]:
# Temperature variable (TT) must be present in the intermediate file
wps_inter_ds = pyw.rinter(wps_inter_filepaths[0])
tt_var = wps_inter_ds["TT"]

tt_geoproj = tt_var.geoinfo
plev = tt_var.level
lat = np.arange(tt_var.val.shape[-2]) * tt_geoproj["DELTALAT"] + tt_geoproj["STARTLAT"]
lon = np.arange(tt_var.val.shape[-1]) * tt_geoproj["DELTALON"] + tt_geoproj["STARTLON"]
ds_target = xr.Dataset(coords={"lat": lat, "lon": lon})

In [5]:
present = "1995-2014".replace("-", "_")
future = "2045-2064".replace("-", "_")
source_ids = [
    "EC-Earth3",
    "MIROC6",
    "MRI-ESM2-0",
    "ACCESS-CM2",
    "IPSL-CM6A-LR",
    "MPI-ESM1-2-HR",
]
experiment = "ssp585"
variables = ["ta", "ua", "va", "hur", "zg", "ts"]
download_dir = Path("Download_CMIP6/Download")

deltas = {}

for variable in variables:
    diffs = []
    for source_id in source_ids:
        present_dir = download_dir / source_id / "historical"
        (present_file,) = present_dir.glob(f"{variable}_*{present}.nc")
        present_ds = xr.open_dataset(present_file)[variable]

        future_dir = download_dir / source_id / experiment
        (future_file,) = future_dir.glob(f"{variable}_*{future}.nc")
        future_ds = xr.open_dataset(future_file)[variable]

        diff = future_ds - present_ds

        regridder = xe.Regridder(diff, ds_target, "bilinear")

        diff = regridder(diff)
        if "plev" in diff.dims:
            diff = diff.interp(plev=plev)

        diffs.append(diff)

    mean_diff = diffs[0].copy(data=np.nanmean(diffs, axis=0))
    mean_diff = mean_diff.fillna(0)

    deltas[variable] = mean_diff

In [6]:
var_mapping = {
    "RH": "hur",
    "TT": "ta",
    "UU": "ua",
    "VV": "va",
    "GHT": "zg",
    "SKINTEMP": "ts",
    "SST": "ts",
}

## Read intermediate and add GWI on these files using pywinter library

In [7]:
for f in wps_inter_filepaths[:]:
    prefix, date = f.name.split(":")
    month = int(date.split("-")[1])

    inter_ds = pyw.rinter(f)

    varnames = list(inter_ds.keys())

    var = inter_ds[varnames[0]]
    geoinfo = pyw.Geo0(
        var.geoinfo["STARTLAT"],
        var.geoinfo["STARTLON"],
        var.geoinfo["DELTALAT"],
        var.geoinfo["DELTALON"],
    )

    fields = []

    for varname in varnames:
        var = inter_ds[varname]
        values = var.val

        # Apply PGW delta
        if varname in ["TT", "UU", "VV", "GHT", "SKINTEMP", "SST"]:
            gwi_val = deltas[var_mapping[varname]].sel(month=month).values
            values = values + gwi_val

        if varname in ["SM", "ST", "SOILM", "SOILT"]:  # soil variables
            field = pyw.Vsl(varname, values, var.level)
        elif isinstance(var.level, str):  # 2D variables
            field = pyw.V2d(
                varname,
                values,
                var.general["DESC"],
                var.general["UNITS"],
                var.general["XLVL"],
            )
        else:
            field = pyw.V3dp(varname, values, var.level)

        fields.append(field)

    pyw.cinter(prefix, date, geoinfo, fields, dst_dir)

ERA5:2017-04-19_00
ERA5:2017-04-19_01
ERA5:2017-04-19_02
ERA5:2017-04-19_03
ERA5:2017-04-19_04
ERA5:2017-04-19_05
ERA5:2017-04-19_06
ERA5:2017-04-19_07
ERA5:2017-04-19_08
ERA5:2017-04-19_09
ERA5:2017-04-19_10
ERA5:2017-04-19_11
ERA5:2017-04-19_12
ERA5:2017-04-19_13
ERA5:2017-04-19_14
ERA5:2017-04-19_15
ERA5:2017-04-19_16
ERA5:2017-04-19_17
ERA5:2017-04-19_18
ERA5:2017-04-19_19
ERA5:2017-04-19_20
ERA5:2017-04-19_21
ERA5:2017-04-19_22
ERA5:2017-04-19_23
